# ARC2 019 P1 pilot
90-min time-boxed plumbing check of the full-submit chain (mount, C8 direct, stage2, pool, atomic writes, cutoff). Same blobs as full submit; not a coverage claim.


In [ ]:
import base64, hashlib, json, os, signal, subprocess, sys, time
import glob as _glob
from pathlib import Path
notebook_started = time.time()
source_dir = Path("/kaggle/working/019-pilot-code")
source_dir.mkdir(exist_ok=False)
blobs = json.loads(base64.b64decode("..."))
expected = {'vdriver019_submit.py': 'b5282e9e06187afd9734c929788994e072070a509d711d18a9328d774c8a818c', 'submit_lib.py': 'f2b4d4db4a30005481d35e60f75ffdc7c6143b75f79419a9c89d44465b236e58', 'run_reasoning_probe.py': '6c76df003b92a7d6b6b6158fbeda659b3ff78457d4a391e456b71488f39ce7a6', 'solve_program.py': '03aaa3818a8f3beafee019140bae823f968e913af72ef297fa7531f24816a37a'}
for name, encoded in blobs.items():
    raw = base64.b64decode(encoded)
    assert hashlib.sha256(raw).hexdigest() == expected[name], name
    dest = source_dir / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(raw)
PY312 = '/usr/bin/python3.12'
assert Path(PY312).is_file(), "python3.12 binary missing"
ver = subprocess.run([PY312, "--version"], capture_output=True, text=True)
print("PY312", (ver.stdout + ver.stderr).strip(), flush=True)
assert "Python 3.12" in (ver.stdout + ver.stderr), "not python 3.12"
cands = sorted(_glob.glob("/kaggle/input/**/*challenge*.json", recursive=True))
print("challenge candidates:", cands[:10], flush=True)
challenges_path = None
arc_roots = sorted(_glob.glob("/kaggle/input/arc-prize-2026-arc-agi-2/*"))
print("competition files:", [str(x)[:120] for x in arc_roots][:20], flush=True)
for c in cands:
    try:
        obj = json.loads(Path(c).read_text())
    except Exception:
        continue
    if isinstance(obj, dict) and obj and all(isinstance(v, dict) and "train" in v and "test" in v for v in list(obj.values())[:5]):
        challenges_path = c
        break
assert challenges_path is not None, "no challenge json found under /kaggle/input"
print("using challenges:", challenges_path, flush=True)
print(json.dumps({"event": "boot", "source_sha256": {k: v[:12] for k, v in expected.items()}, "py312": (ver.stdout + ver.stderr).strip(), "python": sys.version, "notebook_started": notebook_started, "session_seconds": 5400}), flush=True)
proc = subprocess.Popen([PY312, '-u', str(source_dir / 'vdriver019_submit.py'), '--work', '/kaggle/working/vprobe', '--challenges', challenges_path, '--session-seconds', '5400', '--pip-python', sys.executable], cwd=str(source_dir), start_new_session=True)
try:
    rc = proc.wait(timeout=5580)
except subprocess.TimeoutExpired:
    print('NOTEBOOK_DEADLINE', flush=True)
    os.killpg(proc.pid, signal.SIGTERM)
    try:
        rc = proc.wait(timeout=90)
    except subprocess.TimeoutExpired:
        os.killpg(proc.pid, signal.SIGKILL)
        rc = proc.wait(timeout=30)
print('VDRIVER_RETURN_CODE', rc, flush=True)
assert rc == 0, 'Inspect /kaggle/working/vprobe/run.json'
